# Sauda shopping assistant: test on Groq and Gemini

**Before you start:** click the **key icon** on the left (Secrets) and add your keys:

| Name | Value |
|---|---|
| `GEMINI_API_KEY` | your Gemini key |
| `GROQ_API_KEY` | your Groq key |

Turn on **Notebook access** for each. You can add just one.

Then run the cells from top to bottom: press the ▶ button on each cell and wait for it to finish.

## Step 1: Download and install (about 30 seconds)

In [ ]:
!rm -rf Usman-hassan
!git clone -q --depth 1 -b claude/ecommerce-ai-assistant-t27kqw https://github.com/Usmanhassan7654/Usman-hassan.git
%cd Usman-hassan/sauda-assistant
!pip install -q -r requirements.txt
print("Installed.")

## Step 2: Load your keys

In [ ]:
import os
from google.colab import userdata

found = []
for name in ("GEMINI_API_KEY", "GROQ_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        found.append(name)
print("Keys loaded:", ", ".join(found) if found else "none. Add them in the key icon on the left, turn on Notebook access, then run this cell again.")

## Step 3: Check the app itself (no keys needed)
Should end with **passed**.

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## Step 4: Chat with the assistant here
Change the message in the last line and run the cell again. The conversation continues between runs.

In [ ]:
import sys, uuid
sys.path.insert(0, ".")
from sauda import ingest, llm
from sauda.db import Repo
from sauda.agent import Agent

repo = Repo("colab.db")
scope = repo.add_store("demo-mart", "Demo Mart", "demo-khi", "Karachi", api_key="demo-store-key")
if not repo.catalog_stats(scope)["published_products"]:
    ingest.sync(repo, scope, "data/sample_pk_grocery.csv")

# Order of providers: change to "gemini" or "groq" to use only one.
os.environ.setdefault("SAUDA_LLM", ",".join(p for p, k in (("groq", "GROQ_API_KEY"), ("gemini", "GEMINI_API_KEY")) if os.environ.get(k)))
agent = Agent(repo, model=llm.build())
session = uuid.uuid4().hex
print("Using:", agent.model.name)

def ask(message):
    r = agent.handle(scope, session, message, emit=lambda e: print("  …", e["text"]))
    print("\nAssistant:", r.reply, "\n")
    for card in r.cards:
        if card["type"] == "plan":
            print("Suggested list:")
            for l in card["lines"]:
                print(f"  {l['packs']} x {l['name']}  Rs {l['line_total']:,.0f}")
            print(f"  Total: Rs {card['total']:,.0f}")
            for b in card["budgets"]:
                print(f"  Budget {b['budget']:,.0f}: {'OVER by ' + format(-b['remaining'], ',.0f') if b['over'] else format(b['remaining'], ',.0f') + ' left'}")
        elif card["type"] == "options":
            for g in card["groups"]:
                print(g["label"] + ":")
                for o in g["options"]:
                    print(f"  {o['ref']}. {o['name']}  Rs {o['price']:,.0f}")
        elif card["type"] == "comparison":
            for i in card["items"]:
                print(f"  {i['ref']}. {i['name']}  Rs {i['price']:,.0f}  {i['unit_price']}")
    if r.basket["items"]:
        print(f"Basket: {r.basket['item_count']} items, Rs {r.basket['total']:,.0f}")
    if r.grounding.get("corrected"):
        print("(The assistant corrected its first reply:", r.grounding["corrected"], ")")
    print(f"[{r.usage['model_calls']} model calls via {', '.join(r.usage['models'])}]")

ask("chicken biryani banana hai 6 logon ke liye")

In [ ]:
ask("Olpers nahi, aur oil sasta wala do")

In [ ]:
ask("theek hai, sab basket mein daal do")

## Step 5: Run the automatic customer tests
Each provider takes a few minutes and uses some of your API credit. At the end you'll see a line like **8/10 journey runs passed**. Copy everything this cell prints and send it to Claude.

In [ ]:
for provider, key in (("groq", "GROQ_API_KEY"), ("gemini", "GEMINI_API_KEY")):
    if os.environ.get(key):
        print(f"===================== {provider} =====================")
        !python evals/journeys.py --llm {provider} 2>&1 | grep -v "^Report:"
    else:
        print(f"Skipping {provider}: no key.")

## Step 6 (optional): Open the full chat website
Run this, then click the link that appears.

In [ ]:
import subprocess, time
from google.colab import output
env = {**os.environ, "SAUDA_DEMO": "1", "SAUDA_DB": "web.db"}
server = subprocess.Popen(["python", "-m", "uvicorn", "sauda.server:app", "--port", "8000"], env=env,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
output.serve_kernel_port_as_window(8000, anchor_text="Open the Sauda chat website")